# nb_ingestion_generic — the parameterized worker behind the runMultiple DAG
**Platform pattern:** one notebook, N entities. The coordinator (pipeline or `runMultiple` DAG —
Sec 22 of the internals doc) passes `ENTITY_ID`; everything else comes from `etl_entity` metadata.
Incremental loads use a **watermark persisted in the run log** — the notebook is idempotent:
re-running it ingests only what arrived since the last successful run, and running it twice in a
row is safe (proven below).

In [1]:
NOTEBOOK_NAME = "nb_ingestion_generic"
TABLES_ROOT   = "/tmp/fabric_kit_warehouse"
ENTITY_TABLE  = f"{TABLES_ROOT}/_ops/etl_entity"
RUN_LOG       = f"{TABLES_ROOT}/_ops/etl_run_log"
ENTITY_ID     = 1

In [2]:
# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)

# --- session bootstrap (identical in every kit notebook; Fabric supplies `spark`) ---
import os, sys, json, time
from datetime import datetime, timezone

def get_session():
    try:
        return spark  # noqa: F821  (Fabric / existing session)
    except NameError:
        from pyspark.sql import SparkSession
        from delta import configure_spark_with_delta_pip
        b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
             .config("spark.driver.memory", "2g")
             .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
             .config("spark.sql.catalog.spark_catalog",
                     "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
        return configure_spark_with_delta_pip(b).getOrCreate()

spark = get_session()
spark.sparkContext.setLogLevel("ERROR")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
APP_ID = spark.sparkContext.applicationId
print(f"{NOTEBOOK_NAME} | run {RUN_ID} | app {APP_ID} | Spark {spark.version}")

26/08/04 12:15:57 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/04 12:15:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-07c81938-9c24-4264-8a12-184728386312;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central


	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 266ms :: artifacts dl 12ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-07c81938-9c24-4264-8a12-184728386312
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/8ms)


26/08/04 12:15:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1
nb_ingestion_generic | run 20260804T121601Z | app local-1785845760706 | Spark 3.5.1


In [3]:
# --- self-contained demo data (skipped if the root already has tables) ---
from pyspark.sql import functions as F
import os
os.makedirs(TABLES_ROOT, exist_ok=True)
def _has_delta(root):
    return any(os.path.isdir(os.path.join(root, d, "_delta_log")) for d in os.listdir(root)) if os.path.isdir(root) else False
if not _has_delta(TABLES_ROOT):
    (spark.range(0, 100_000)
        .withColumn("customer_id", (F.col("id") % 500).cast("int"))
        .withColumn("amount", F.round(F.rand() * 500, 2))
        .withColumn("status", F.when(F.col("id") % 7 == 0, "cancelled").otherwise("complete"))
        .withColumn("order_date", F.date_add(F.lit("2026-06-01"), (F.col("id") % 60).cast("int")))
        .repartition(24)  # deliberately many small files so the audit has something to find
        .write.format("delta").mode("overwrite").save(f"{TABLES_ROOT}/orders"))
    spark.sql(f"""CREATE TABLE IF NOT EXISTS delta.`{TABLES_ROOT}/orders_silver`
        (order_id BIGINT, customer_id INT, amount DOUBLE, status STRING, order_date DATE)
        USING DELTA TBLPROPERTIES('delta.enableDeletionVectors'='true','delta.enableChangeDataFeed'='true')""")
    src = spark.read.format("delta").load(f"{TABLES_ROOT}/orders").withColumnRenamed("id","order_id")
    src.write.format("delta").mode("append").save(f"{TABLES_ROOT}/orders_silver")
    spark.sql(f"DELETE FROM delta.`{TABLES_ROOT}/orders_silver` WHERE status='cancelled'")
    print("demo tables created: orders (24 small files), orders_silver (DV+CDF, has deletes)")
else:
    print("existing tables found - demo data skipped")

existing tables found - demo data skipped


In [4]:
# Seed demo entity metadata once (production: Fabric SQL Database, queried via pyodbc + Entra token).
import os
from pyspark.sql import functions as F
if not os.path.isdir(os.path.join(ENTITY_TABLE, "_delta_log")):
    entities = [(1, json.dumps({
        "source_path": f"{TABLES_ROOT}/orders",
        "target_path": f"{TABLES_ROOT}/silver_orders_inc",
        "load_type": "incremental",
        "merge_keys": ["order_id"],
        "watermark_col": "order_date",
        "rename": {"id": "order_id"},
        "target_props": {"delta.enableDeletionVectors": "true",
                          "delta.enableChangeDataFeed": "true"}}))]
    spark.createDataFrame(entities, ["entity_id","config_json"]) \
         .write.format("delta").mode("overwrite").save(ENTITY_TABLE)
    print("demo etl_entity seeded")

In [5]:
from delta.tables import DeltaTable

def last_watermark(entity_id):
    if not os.path.isdir(os.path.join(RUN_LOG, "_delta_log")): return None
    row = (spark.read.format("delta").load(RUN_LOG)
           .where((F.col("entity_id") == entity_id) & (F.col("status") == "OK"))
           .agg(F.max("watermark_to")).collect()[0][0])
    return row

def run_entity(entity_id):
    cfg = json.loads(spark.read.format("delta").load(ENTITY_TABLE)
                     .where(F.col("entity_id") == entity_id).collect()[0]["config_json"])
    t0 = time.time()
    src = spark.read.format("delta").load(cfg["source_path"])
    for old, new in cfg.get("rename", {}).items():
        src = src.withColumnRenamed(old, new)
    wm_col, wm_prev = cfg.get("watermark_col"), last_watermark(entity_id)
    if cfg["load_type"] == "incremental" and wm_prev is not None:
        src = src.where(F.col(wm_col) > F.lit(wm_prev))       # only what arrived since last OK run
    rows_in = src.count()
    wm_new = src.agg(F.max(wm_col)).collect()[0][0] if rows_in else wm_prev

    if not os.path.isdir(os.path.join(cfg["target_path"], "_delta_log")):
        w = src.write.format("delta")
        for k, v in cfg["target_props"].items(): w = w.option(k, v)
        w.save(cfg["target_path"])
    elif rows_in:
        on = " AND ".join(f"t.{k} = s.{k}" for k in cfg["merge_keys"])   # full-key MERGE hygiene
        (DeltaTable.forPath(spark, cfg["target_path"]).alias("t")
            .merge(src.alias("s"), on).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())

    entry = {"entity_id": entity_id, "run_id": RUN_ID, "status": "OK", "rows_in": rows_in,
             "watermark_from": str(wm_prev), "watermark_to": str(wm_new),
             "secs": round(time.time() - t0, 1), "app_id": APP_ID}
    spark.createDataFrame([entry]).write.format("delta").mode("append") \
         .option("mergeSchema", "true").save(RUN_LOG)
    return entry

first = run_entity(ENTITY_ID)
print("run 1:", first)
second = run_entity(ENTITY_ID)
print("run 2:", second)
assert second["rows_in"] == 0, "idempotency broken: second run should ingest nothing new"
tgt = spark.read.format("delta").load(f"{TABLES_ROOT}/silver_orders_inc").count()
print(f"idempotency proven - run 2 ingested 0 rows; target holds {tgt} rows")
spark.stop() if "local" in spark.sparkContext.master else None

run 1: {'entity_id': 1, 'run_id': '20260804T121601Z', 'status': 'OK', 'rows_in': 0, 'watermark_from': '2026-07-30', 'watermark_to': '2026-07-30', 'secs': 9.4, 'app_id': 'local-1785845760706'}


run 2: {'entity_id': 1, 'run_id': '20260804T121601Z', 'status': 'OK', 'rows_in': 0, 'watermark_from': '2026-07-30', 'watermark_to': '2026-07-30', 'secs': 4.6, 'app_id': 'local-1785845760706'}


idempotency proven - run 2 ingested 0 rows; target holds 100000 rows
